# Импорты

In [1]:
from collections.abc import Callable
from typing import List

import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Математическая модель

Уравнение: <br>
$
\begin{cases}
    \displaystyle v' = -\frac{2}{m} \\
    v(400) = 0
\end{cases}
$

Решение: <br>
$\displaystyle
v = 2\ln \frac{400}{m}
$

In [2]:
m_0 = 50
m_n = 400

v_n = 0

def f(m: float, _v: float) -> float:
    return -2 / m

def v(m: float) -> float:
    return 2 * np.log(m_n / m)

# Решение

Решим уравнение методом Рунге-Кутта 4 порядка, но в обратном направлении:<br>
$\displaystyle
v_{i - 1} = v_i - \frac{h}{6} (k_1 + 2k_2 + 2k_3 + k_4) \\
k_1 = f(m_i, v_i) \\
k_2 = f \left( m_i - \frac{h}{2}, v_i - \frac{h}{2} k_1 \right) \\
k_3 = f \left( m_i - \frac{h}{2}, v_i - \frac{h}{2} k_2 \right) \\
k_4 = f(m_i - h, v_i - h k_3)
$

In [3]:
def runge_kutta_4_reverse(n: int, f: Callable[[float, float], float], m: List[float]) -> List[float]:
    h = (m_n - m_0) / n
    y_numeric = [0.0] * (n + 1)
    y_numeric[-1] = v_n

    for i in range(n, 0, -1):
        k_1 = f(m[i], y_numeric[i])
        k_2 = f(m[i] - h / 2, y_numeric[i] - h / 2 * k_1)
        k_3 = f(m[i] - h / 2, y_numeric[i] - h / 2 * k_2)
        k_4 = f(m[i] - h, y_numeric[i] - h * k_3)

        y_numeric[i - 1] = y_numeric[i] - h / 6 * (k_1 + 2 * k_2 + 2 * k_3 + k_4)

    return y_numeric

# Графики

In [13]:
def compare_solutions(n: int, f: Callable[[float, float], float], v: Callable[[float], float]):
    m = np.linspace(m_0, m_n, n + 1)

    v_exact = np.array([v(x_i) for x_i in m])
    v_numeric = np.array(runge_kutta_4_reverse(n, f, m))
    error = np.abs(v_exact - v_numeric)

    figure = make_subplots(rows=1, cols=2)

    figure.add_trace(
        go.Scatter(x=m, y=v_exact, mode="lines", name="Точное", legend="legend"),
        row=1, col=1,
    )
    figure.add_trace(
        go.Scatter(x=m, y=v_numeric, mode="lines", name="Численное", legend="legend"),
        row=1, col=1,
    )

    figure.add_trace(
        go.Scatter(x=m, y=error, mode="lines", name="Погрешность", legend="legend2"),
        row=1, col=2,
    )

    figure.update_yaxes(exponentformat="power", row=1, col=1)
    figure.update_yaxes(exponentformat="power", row=1, col=2)
    figure.update_xaxes(title_text="m", row=1, col=1)
    figure.update_xaxes(title_text="m", row=1, col=2)
    figure.update_yaxes(title_text="v", row=1, col=1)
    figure.update_yaxes(title_text="error", row=1, col=2)

    figure.update_layout(
        height=600,
        legend=dict(
            orientation="v", 
            xanchor="right", 
            yanchor="top", 
            x=0.45,
        ),
        legend2=dict(
            orientation="v", 
            xanchor="right", 
            yanchor="top", 
            x=1,
        ),
    )
    figure.show()

ns = [10, 100, 1000]
for n in ns:
    compare_solutions(n, f, v)